# Занятие 22. Практика: классификация иконок методом k ближайших соседей (kNN)

Вы **пишете код сами** в пустых ячейках. Блоки **«Легенда»** и **«Дано»** не меняйте.

Теория — в `Урок_21_Workflow_kNN.ipynb`. Проверочные критерии прямо указаны в заданиях; решение преподаватель может хранить отдельно.

### Оценивание (28 баллов)

| № | Тема | Баллы |
|---|------|------:|
| 1 | Мини-симуляция релиза иконок | 2 |
| 2 | Импорты и разбиение данных | 4 |
| 3 | Осмотр train (числа) | 2 |
| 4 | Диаграммы по train | 3 |
| 5 | Baseline | 3 |
| 6 | kNN и сравнение с baseline | 3 |
| 7 | Масштабирование и kNN | 3 |
| 8 | Подбор `k` и график | 3 |
| 9 | Матрица ошибок и recall | 3 |
| 10 | Финальная проверка на test | 2 |
| | **Итого** | **28** |


---
## Легенда: студия «Pixel Quest»

Вы — стажёр в инди-студии, которая выпускает пошаговую стратегию **«Pixel Quest»**.
Игрок расставляет на карте **юниты** — крошечные чёрно-белые фигурки размером **10×10 пикселей**.

### Три типа юнитов

| Класс | По-русски | Роль в игре |
|-------|-----------|-------------|
| `castle` | замок | защита базы, «тяжёлая» фигура |
| `dragon` | дракон | летающий атакующий юнит |
| `knight` | рыцарь | пеший боец |

Игроки рисуют свои варианты иконок и присылают в студию. Художники и геймдизайнеры
отобрали **229** удачных рисунков и сохранили в `approved.json`.
Поле `initial` — правильный тип юнита.

### Зачем нужен этот датасет

Это **эталонная коллекция**: на ней мы учим модель понимать, «похожа» новая иконка на замок, дракона или рыцаря.

### Зачем задача классификации

Когда приходит **новый** рисунок, игра должна **автоматически** подсказать тип и поставить иконку
на нужную кнопку в интерфейсе. Если модель перепутает дракона с рыцарем — игрок увидит не ту фигурку,
а это ломает тактику и доверие к игре.

Ваша цель на занятии — пройти полный путь ML-разработки и обучить **kNN**,
который по 100 пикселям угадывает класс юнита.

### Короткий словарь

| Слово | Значение |
|-------|----------|
| **признак** | одно число про иконку (здесь: один пиксель 0 или 1) |
| **метка / класс** | правильный ответ: `castle`, `dragon`, `knight` |
| **train** | обучающая выборка — на ней модель **учится** |
| **validation** | проверочная — на ней **сравниваем** варианты (например, разные `k`) |
| **test** | отложенная — **финальный экзамен** (один раз в конце) |
| **baseline** | наивная модель «для сравнения» |
| **accuracy** | доля верных ответов |


---
## Дано: датасет и картинки

Каждая иконка — матрица 10×10. В таблице `X_all` она записана **строкой из 100 чисел**
(столбцы `p0` … `p99`). В `y_all` — метка класса.

### Как выглядят иконки разных классов

![Примеры трёх классов](practice_images/samples_by_class.png)

По одному крупному примеру:

| Замок | Дракон | Рыцарь |
|-------|--------|--------|
| ![castle](practice_images/sample_castle.png) | ![dragon](practice_images/sample_dragon.png) | ![knight](practice_images/sample_knight.png) |

### Весь датасет целиком

![Обзор датасета](practice_images/dataset_overview.png)

![Датасет по классам](practice_images/dataset_by_class.png)

### Справочные диаграммы (по всему датасету)

Их **не нужно** перерисовывать в заданиях — это ориентир. В заданиях 3–4 вы построите похожие графики **только по train**.

![Число иконок по классам](practice_images/chart_class_counts.png)
![Плотность пикселей по классам](practice_images/chart_density_boxplot.png)
![Нижний ряд пикселей](practice_images/chart_bottom_row.png)

Запустите ячейку ниже, чтобы загрузить данные в `X_all` и `y_all`.


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

DATA_PATH = Path('approved.json')
items = json.loads(DATA_PATH.read_text(encoding='utf-8'))

rows, labels = [], []
for item in items:
    matrix = np.array(item['data']['pixels'], dtype=int)
    rows.append(matrix.flatten())
    labels.append(item['data']['initial'])

X_all = pd.DataFrame(rows, columns=[f'p{i}' for i in range(100)])
y_all = pd.Series(labels, name='label')

print('Объектов:', X_all.shape[0], '| признаков:', X_all.shape[1])
print('Классы:')
print(y_all.value_counts())
X_all.iloc[:, :8].head()


---
## Задание 1. Мини-симуляция релиза иконок — **2 балла**

Представьте, что в игру пришла новая иконка. Нужно решить:
- показать её автоматически;
- или отправить художнику на ручную проверку.

В этом задании модель ещё не обучена, поэтому `model_confidence` — **условная уверенность прототипа**, которую вы задаёте сами числами от 0 до 1. Это маленькая разминка про две вещи: как хранить простое правило релиза и как применять его к таблице новых иконок.

### Что сделать

**Шаг 1.** Создайте словарь `release_rule`. В нём должно быть 3 настройки:
- `auto_accept_threshold` — порог уверенности от 0 до 1;
- `auto_action` — текст решения для уверенных иконок;
- `manual_action` — текст решения для неуверенных иконок.

**Шаг 2.** Создайте `incoming_icons` (4 строки) с колонками:
`icon_id`, `predicted_class`, `model_confidence`.

**Шаг 3.** Добавьте в `incoming_icons` колонку `decision`. Для каждой строки сравните `model_confidence` с порогом из `release_rule`:
- если `model_confidence >= auto_accept_threshold`, запишите значение из `auto_action`;
- иначе запишите значение из `manual_action`.

**Шаг 4.** Зафиксируйте метрику будущей модели: `PRIMARY_METRIC = 'accuracy'`. В markdown напишите 1–2 предложения: почему для неуверенных иконок всё равно нужна ручная проверка, даже если основная метрика модели — accuracy.

### Подробные критерии (для проверки LLM)

- **0.5 балла** — в `release_rule` есть все 3 ключа и `auto_accept_threshold` в диапазоне `(0, 1)`.
- **0.5 балла** — `incoming_icons` содержит минимум 4 строки и нужные 3 колонки.
- **0.5 балла** — `decision` заполнена по правилу порога (`>=` → авто, `<` → ручная проверка).
- **0.5 балла** — задан `PRIMARY_METRIC = 'accuracy'` и есть прикладное объяснение в markdown.

### Снижение баллов
- Нет `decision` или она не зависит от порога → не выше **1.0**.
- Нет `PRIMARY_METRIC` или нет объяснения → минус **0.5**.


In [ ]:
release_rule = {
    'auto_accept_threshold': 0.75,
    'auto_action': 'показать автоматически',
    'manual_action': 'отправить на ручную проверку',
}

incoming_icons = pd.DataFrame(
    {
        'icon_id': [101, 102, 103, 104],
        'predicted_class': ['castle', 'dragon', 'knight', 'dragon'],
        'model_confidence': [0.91, 0.82, 0.61, 0.48],
    }
)

incoming_icons['decision'] = np.where(
    incoming_icons['model_confidence'] >= release_rule['auto_accept_threshold'],
    release_rule['auto_action'],
    release_rule['manual_action'],
)

PRIMARY_METRIC = 'accuracy'
incoming_icons

*Почему нужна ручная проверка?*

Ручная проверка нужна для иконок с низкой уверенностью, потому что ошибка в автоматическом решении сразу попадёт в интерфейс игры. Accuracy показывает среднее качество модели, но отдельная неуверенная иконка всё равно может оказаться важной или необычной, поэтому её безопаснее отправить художнику.


---
## Задание 2. Импорты и разбиение данных — **4 балла**

### Что сделать

**Шаг 1.** Подключите импорты: `train_test_split`, `StandardScaler`, `KNeighborsClassifier`, `DummyClassifier`, `accuracy_score`, `recall_score`, `confusion_matrix`, `matplotlib.pyplot as plt`.

**Шаг 2.** Задайте константы для воспроизводимого разбиения:
- `RANDOM_STATE = 42`
- `TEST_SIZE = 0.2`
- `VAL_SIZE = 0.25`

**Шаг 3.** Сделайте первый split: отделите test от `X_all, y_all`. Остальные данные сохраните как временную часть train+validation.

**Шаг 4.** Сделайте второй split: из временной части train+validation выделите validation, а оставшееся оставьте для train.

**Шаг 5.** Выведите размеры train/validation/test.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — есть все нужные импорты.
- **1.0 балл** — заданы константы `RANDOM_STATE`, `TEST_SIZE`, `VAL_SIZE`.
- **1.0 балл** — выполнены 2 split со `stratify`, получены `X_train`, `X_val`, `X_test`, `y_train`, `y_val`, `y_test`.
- **1.0 балл** — размеры `137 / 46 / 46`.

### Снижение баллов
- Нет `stratify` в split → минус **0.5** за каждый случай.
- train/val/test не выделены явно → минус **0.5**.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt
RANDOM_STATE, TEST_SIZE, VAL_SIZE = 42, 0.2, 0.25

X_train_val, X_test, y_train_val, y_test = train_test_split(X_all, y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_all)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=VAL_SIZE, random_state=RANDOM_STATE, stratify=y_train_val)
print(len(X_train), len(X_val), len(X_test))

---
## Задание 3. Осмотр train (числа) — **2 балла**

### Что сделать

**Шаг 1.** Выведите распределение классов: `y_train.value_counts()`.

**Шаг 2.** Создайте `train_df` как копию `X_train`: это таблица признаков `p0` … `p99` только для train. Затем добавьте в неё колонку `label` из `y_train`.

**Шаг 3.** Посчитайте `pixel_density = сумма 100 пикселей / 100`.

**Шаг 4.** Выведите среднюю `pixel_density` по классам.

### Подробные критерии (для проверки LLM)

- **0.5 балла** — выведено распределение классов в `y_train`.
- **0.5 балла** — создан `train_df` на основе `X_train` и добавлена колонка `label` из `y_train`.
- **0.5 балла** — рассчитана `pixel_density`.
- **0.5 балла** — выведена средняя `pixel_density` по классам.

### Снижение баллов
- Использованы val/test вместо train → минус **0.5**.
- `pixel_density` без деления на 100 → минус **0.5**.


In [ ]:
train_df = X_train.copy()
train_df['label'] = y_train.values
train_df['pixel_density'] = train_df.filter(regex='^p').sum(axis=1) / 100
print(y_train.value_counts())
print(train_df.groupby('label', observed=True)['pixel_density'].mean().round(3))

---
## Задание 4. Диаграммы по train — **3 балла**

### Что сделать

Постройте 3 графика по train:

**Шаг 1.** Bar: число объектов каждого класса.

**Шаг 2.** Bar: средняя `pixel_density` по классам.

**Шаг 3.** Boxplot: распределение `pixel_density` по классам.

У каждого графика должны быть `title`, `xlabel`, `ylabel`, `plt.show()`.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — bar «Классы в train».
- **1.0 балл** — bar средней `pixel_density`.
- **1.0 балл** — boxplot `pixel_density`.

### Снижение баллов
- Нет title/подписей осей → минус **0.25** за элемент.
- Графики по `X_all`, а не train → минус **0.5**.


In [ ]:
counts = y_train.value_counts()
plt.figure(figsize=(5,3)); plt.bar(counts.index, counts.values)
plt.title('Классы в train'); plt.xlabel('класс'); plt.ylabel('количество'); plt.show()
means = train_df.groupby('label', observed=True)['pixel_density'].mean()
plt.figure(figsize=(5,3)); plt.bar(means.index, means.values)
plt.title('Средняя плотность пикселей в train'); plt.xlabel('класс'); plt.ylabel('плотность'); plt.show()
data = [train_df.loc[train_df.label==c, 'pixel_density'] for c in ['castle','dragon','knight']]
plt.figure(figsize=(5,3)); plt.boxplot(data, tick_labels=['castle','dragon','knight'])
plt.title('Разброс плотности по классам (train)'); plt.ylabel('pixel_density'); plt.show()

---
## Задание 5. Baseline — **3 балла**

### Что сделать

**Шаг 1.** Обучите `DummyClassifier(strategy='most_frequent')` на train.

**Шаг 2.** Посчитайте `dummy_acc` на validation.

**Шаг 3.** Найдите самый частый класс в train и его долю.

**Шаг 4.** Сравните `dummy_acc` с долей самого частого класса. Они должны быть близки по смыслу: baseline всегда выбирает самый частый класс.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — baseline обучен.
- **1.0 балл** — `dummy_acc` посчитан на validation.
- **1.0 балл** — есть сравнение с долей самого частого класса в train.

### Снижение баллов
- Метрика не на validation → минус **1.0**.
- Нет сравнения с самым частым классом → минус **0.5**.


In [ ]:
dummy = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
dummy_acc = accuracy_score(y_val, dummy.predict(X_val))

most_frequent_class = y_train.value_counts().idxmax()
most_frequent_share = y_train.value_counts(normalize=True).max()

print('dummy_acc', round(dummy_acc, 3))
print('самый частый класс в train:', most_frequent_class)
print('его доля в train:', round(most_frequent_share, 3))


---
## Задание 6. kNN и сравнение с baseline — **3 балла**

### Что сделать

**Шаг 1.** Обучите `KNeighborsClassifier(n_neighbors=5)` на `X_train`, `y_train`.

**Шаг 2.** Посчитайте `acc_raw` на validation.

**Шаг 3.** Постройте bar-диаграмму `dummy_acc` vs `acc_raw`.

**Шаг 4.** Проверьте, что `acc_raw > dummy_acc`.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — обучен kNN на train.
- **1.0 балл** — рассчитан `acc_raw`.
- **1.0 балл** — есть bar-диаграмма сравнения и проверка `acc_raw > dummy_acc`.

### Снижение баллов
- Обучение на `X_train_s` в этом задании → минус **0.5**.
- Нет диаграммы или сравнения → минус **0.5**.


In [ ]:
knn_raw = KNeighborsClassifier(5).fit(X_train, y_train)
acc_raw = accuracy_score(y_val, knn_raw.predict(X_val))
plt.figure(figsize=(4,3)); plt.bar(['baseline','kNN'], [dummy_acc, acc_raw])
plt.title('Baseline vs kNN (validation)'); plt.ylabel('accuracy'); plt.ylim(0,1); plt.show()
print(round(acc_raw,3), acc_raw>dummy_acc)

---
## Задание 7. Масштабирование и kNN — **3 балла**

### Что сделать

**Шаг 1.** `scaler = StandardScaler()`.

**Шаг 2.** `scaler.fit(X_train)`.

**Шаг 3.** Получите `X_train_s`, `X_val_s`, `X_test_s` через `transform`.

**Шаг 4.** Обучите kNN (`n_neighbors=5`) на `X_train_s`.

**Шаг 5.** Посчитайте `acc_scaled` на validation.

**Шаг 6.** Постройте bar `acc_raw` vs `acc_scaled` и дайте краткий вывод в markdown.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — корректное масштабирование (fit только на train).
- **1.0 балл** — посчитан `acc_scaled`.
- **1.0 балл** — есть bar-диаграмма и markdown-вывод.

### Снижение баллов
- `fit` на val/test → минус **1.0**.
- Нет markdown-вывода → минус **0.5**.


In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

acc_scaled = accuracy_score(y_val, KNeighborsClassifier(5).fit(X_train_s,y_train).predict(X_val_s))
plt.figure(figsize=(4,3)); plt.bar(['acc_raw','acc_scaled'], [acc_raw, acc_scaled])
plt.title('Эффект масштабирования'); plt.ylabel('accuracy'); plt.ylim(0,1); plt.show()
print(round(acc_raw,3), round(acc_scaled,3))

*Вывод про масштабирование:*

Для kNN масштабирование обычно важно: модель сравнивает объекты по расстояниям, и признаки с большим разбросом могут начать влиять сильнее остальных. Если `acc_scaled` выше или не хуже `acc_raw`, масштабированную версию стоит оставить как более корректную для метода ближайших соседей.


---
## Задание 8. Подбор `k` и график — **3 балла**

### Что сделать

**Шаг 1.** Переберите `k` из `[1, 3, 5, 7, 11, 15, 21]`.

**Шаг 2.** Соберите таблицу `k_results` с колонками `k`, `val_acc`.

**Шаг 3.** Найдите `best_k` по validation accuracy.

**Шаг 4.** Постройте линейный график `k` → `val_acc` и отметьте `best_k`.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — есть перебор всех `k`.
- **1.0 балл** — есть `k_results` и `best_k`.
- **1.0 балл** — есть график с отмеченной лучшей точкой.

### Снижение баллов
- Подбор `k` по test → минус **1.0**.
- Нет отметки `best_k` на графике → минус **0.5**.


In [ ]:
rows=[]
for k in [1,3,5,7,11,15,21]:
    m=KNeighborsClassifier(k).fit(X_train_s,y_train)
    rows.append({'k':k,'val_acc':round(accuracy_score(y_val,m.predict(X_val_s)),3)})
k_results=pd.DataFrame(rows); best_k=int(k_results.loc[k_results.val_acc.idxmax(),'k'])
display(k_results)
print('best_k:', best_k)

plt.figure(figsize=(6,3))
plt.plot(k_results['k'], k_results['val_acc'], 'o-')
best_acc = k_results.loc[k_results['k']==best_k,'val_acc'].iloc[0]
plt.scatter([best_k],[best_acc], s=120, c='red', label=f'best k={best_k}')
plt.xlabel('k'); plt.ylabel('val_acc'); plt.title('Подбор числа соседей'); plt.legend(); plt.show()


---
## Задание 9. Матрица ошибок и recall — **3 балла**

### Что сделать

**Шаг 1.** Обучите kNN с `best_k` на `X_train_s`, получите `y_pred` на validation.

**Шаг 2.** Посчитайте `confusion_matrix` с фиксированным порядком классов:
`['castle', 'dragon', 'knight']`.

**Шаг 3.** Постройте тепловую карту матрицы ошибок.

**Шаг 4.** Посчитайте recall по классам (`average=None`) и отдельно `recall_dragon`.

**Шаг 5.** В markdown объясните, что означает recall для класса `dragon`: какие ошибки он показывает и почему это важно для игры.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — `confusion_matrix` посчитана с фиксированным порядком классов и построена тепловая карта.
- **1.0 балл** — есть recall по классам и `recall_dragon`.
- **1.0 балл** — есть текстовое объяснение смысла recall для класса `dragon`.

### Снижение баллов
- Нет фиксированного порядка классов → минус **0.5**.
- Нет тепловой карты → минус **0.5**.
- Нет объяснения recall → минус **0.5**.


In [ ]:
labels_order=['castle','dragon','knight']
final=KNeighborsClassifier(best_k).fit(X_train_s,y_train); y_pred=final.predict(X_val_s)
cm=confusion_matrix(y_val,y_pred,labels=labels_order); print(cm)

plt.figure(figsize=(4,3)); plt.imshow(cm, cmap='Blues')
plt.xticks(range(3), labels_order); plt.yticks(range(3), labels_order)
plt.xlabel('прогноз'); plt.ylabel('истина'); plt.title('Матрица ошибок (validation)')
for i in range(3):
    for j in range(3): plt.text(j,i,cm[i,j],ha='center',va='center')
plt.show()

recalls = recall_score(y_val,y_pred,labels=labels_order,average=None)
recall_dragon = recalls[1]
plt.figure(figsize=(5,3)); plt.bar(labels_order, recalls)
plt.title('Recall по классам'); plt.ylabel('recall'); plt.ylim(0,1); plt.show()
print('recall_dragon', round(recall_dragon,3))


*Что означает recall для dragon?*

Recall для `dragon` показывает, какую долю настоящих драконов модель смогла найти среди всех иконок, где правильный класс — `dragon`. Если `recall_dragon` низкий, значит часть драконов модель ошибочно отправляет в другие классы, например в `knight` или `castle`. Для игры это важно: игрок ожидает увидеть дракона как летающего атакующего юнита, а пропущенный дракон ломает смысл иконки в интерфейсе.


---
## Задание 10. Финальная проверка на test — **2 балла**

### Что сделать

**Шаг 1.** Посчитайте `test_acc` на `X_test_s` после выбора `best_k`.

**Шаг 2.** Для первой тестовой иконки получите `pred_new` после `scaler.transform(...)`.

**Шаг 3.** Сравните `pred_new` с истинной меткой `y_test.iloc[0]`.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — корректно посчитан `test_acc` (test не используется для подбора).
- **1.0 балл** — есть `pred_new` и сравнение с `y_test.iloc[0]`.

### Снижение баллов
- Использование test для подбора параметров → минус **1.0**.
- Нет сравнения с истинной меткой → минус **0.5**.


In [ ]:
test_acc = accuracy_score(y_test, final.predict(X_test_s))
print('test_acc', round(test_acc,3))

pred_new = final.predict(scaler.transform(X_test.iloc[[0]]))[0]
print('прогноз', pred_new, 'истина', y_test.iloc[0])
icon = X_test.iloc[0].to_numpy().reshape(10,10)
plt.figure(figsize=(2,2)); plt.imshow(icon, cmap='gray_r', vmin=0, vmax=1); plt.title('Тестовая иконка'); plt.axis('off'); plt.show()

---
## Итог

Вы прошли путь от легенды и картинок до модели, которая помогает студии «Pixel Quest»
распознавать тип новой иконки.
